# Responses API streaming in Jupyter

This notebook shows the compact default transcript, category filtering, silent consumption, and opt-in protocol details. The examples force Web Search and Code Interpreter so their progress events and streamed Python are easy to see.

Install `ai-studio-helpers`, then set lowercase `folder_id` and `api_key` environment variables.

In [3]:
import sys
sys.path.append("../../src")
import os

from openai import OpenAI
from yhelpers.responses.streaming import jstream

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)
model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8"

## Web Search with the compact default

With no flags, `jstream` shows useful activity—such as “Web search”, the query, citations, and the answer—but hides response lifecycle bookkeeping, token usage, and diagnostic JSON.

In [4]:
web_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use web search. Cite the source URL and clearly separate "
        "the verified fact from your explanation."
    ),
    input=(
        "Find the title currently shown on the Python 3 documentation home "
        "page. Then explain in one sentence what that page is for."
    ),
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
web_response = jstream(web_stream)
print("Response:", web_response.id, web_response.status)

Verified fact: Заголовок на главной странице документации Python 3 — «3.14.7 Documentation» (источник: https://docs.python.org/ru/3/).

Объяснение: Эта страница служит центральным справочным ресурсом по языку Python 3, предоставляя официальную документацию, включая руководства, описание синтаксиса, стандартной библиотеки и инструментов.

Response: 77f2dab1-b2f1-4750-9f69-25dc3c24e74c completed


## Code Interpreter with selected categories

This call shows only answer text, Python execution, and errors. Generated Python remains complete and is finalized as a syntax-highlighted `python` fence.

In [5]:
code_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use Code Interpreter. Show the exact Python code, report "
        "the numerical result, and explain the formula in one sentence."
    ),
    input="Calculate the sum of the squares from 1 through 100 with Python.",
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"},
        }
    ],
    stream=True,
)
code_response = jstream(
    code_stream,
    events={"text", "code", "errors"},
)
print("Response:", code_response.id, code_response.status)

I'll calculate the sum of squares from 1 to 100 using Python.

The formula for this calculation is: sum = 1² + 2² + 3² + ... + 100²

Let me execute this calculation:


**Code Interpreter**

```python
try:
    # Calculate the sum of squares from 1 to 100
    sum_of_squares = sum(i**2 for i in range(1, 101))
    
    # Print the result
    print(f"Sum of squares from 1 to 100: {sum_of_squares}")
    
except Exception as e:
    print(f"Error: {str(e)}")
```

The sum of squares from 1 to 100 is 338,350.

The formula used was: sum = 1² + 2² + 3² + ... + 100², which was calculated using Python to sum the squares of each integer from 1 to 100.

Response: 980beae0-83f8-4f8a-86dd-53ebeaa7cbac completed


## Silent consumption

An empty event collection still consumes the complete stream and returns the final `Response`, but displays nothing while it runs.

In [6]:
silent_stream = client.responses.create(
    model=model,
    input="Reply with exactly: stream consumed",
    stream=True,
)
silent_response = jstream(silent_stream, events=[])
print(silent_response.output_text)

stream consumed


## Full protocol diagnostics (opt in)

`events="all"` includes lifecycle and usage events. `show_details=True` adds event names and bounded JSON payloads. Use this combination when diagnosing SDK or provider behavior; it is intentionally more verbose than the default.

In [7]:
debug_stream = client.responses.create(
    model=model,
    instructions="Use web search once, then answer with one sourced sentence.",
    input="What is the official Python documentation URL?",
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
debug_response = jstream(
    debug_stream,
    events="all",
    show_details=True,
    max_chars=1000,
)
print("Debug response:", debug_response.id)

Официальный сайт документации Python находится по адресу https://docs.python.org.

Debug response: a3131274-52a0-4041-9961-5b1ead08dc3c


In [ ]:
client.close()